# 第11章　セグメンテーション（segmentation）とnnU-Net ― 病変を形まで塗り分ける

**『医療診断支援AI開発　基礎編 ― 自分で作る（基礎編）』のコード**

本文に載っているコードを、章の順にそのまま収めています。紙面のコードは読んで理解するためのもの、こちらは動かすためのものです。

- Python 以外（シェル・YAML・Dockerfile など）は、実行環境が違うので**コードセルにせず、そのまま読める形で置いています**。使う場所を確かめてから実行してください。
- 抜粋である以上、上から順に実行するだけで通るとは限りません。データの取得先やパスは、お手元の環境に合わせてください。
- **教育・研究のためのコードです。患者データをこのノートブックに置かないでください。**

リポジトリ: https://github.com/kewel-corp/book-basic

## 11.8　nnU-Netの使い方

```bash
# 1. データセットの前処理と最適設定の自動決定
nnUNetv2_plan_and_preprocess -d 001 --verify_dataset_integrity

# 2. 学習（5分割交差検証の各foldを学習）
nnUNetv2_train 001 3d_fullres 0   # fold 0 を学習
nnUNetv2_train 001 3d_fullres 1   # fold 1 …（0〜4まで）

# 3. 推論
# -f は推論に使うfold。既定は 0 1 2 3 4（全fold）なので、学習していないfoldがあるとここで止まる
nnUNetv2_predict -i 入力フォルダ -o 出力フォルダ -d 001 -c 3d_fullres -f 0 1 2 3 4
```

## 11.10　コードで動かす ― PyTorchで小さなU-Net

In [ ]:
import torch.nn as nn
import torch, torch.nn as nn

def conv_block(cin, cout):                     # 畳み込み2回のひとかたまり
    return nn.Sequential(
        nn.Conv2d(cin, cout, 3, padding=1), nn.BatchNorm2d(cout), nn.ReLU(),
        nn.Conv2d(cout, cout, 3, padding=1), nn.BatchNorm2d(cout), nn.ReLU())

class TinyUNet(nn.Module):
    def __init__(self, n_class=3):
        super().__init__()          # 基底クラス(nn.Module)側の初期化を先に済ませる。省くと層が登録されない
        self.d1 = conv_block(1, 32); self.d2 = conv_block(32, 64)   # エンコーダ
        self.pool = nn.MaxPool2d(2)
        self.b   = conv_block(64, 128)                              # 最深部（1/4解像度）
        self.up2 = nn.ConvTranspose2d(128, 64, 2, stride=2)        # 拡大：1/4→1/2
        self.u2  = conv_block(128, 64)                             # c2 とスキップ結合
        self.up1 = nn.ConvTranspose2d(64, 32, 2, stride=2)         # 拡大：1/2→元の解像度
        self.u1  = conv_block(64, 32)                              # c1 とスキップ結合
        self.out = nn.Conv2d(32, n_class, 1)                       # 画素ごとにクラス出力
    def forward(self, x):
        c1 = self.d1(x); c2 = self.d2(self.pool(c1))               # c1:H  c2:H/2
        b  = self.b(self.pool(c2))                                # b :H/4（プーリング2回）
        u  = self.up2(b)                                          # H/4→H/2
        u  = self.u2(torch.cat([u, c2], dim=1))                    # ← スキップ接続（H/2）
        u  = self.up1(u)                                          # H/2→H（アップサンプリング2回）
        u  = self.u1(torch.cat([u, c1], dim=1))                    # ← スキップ接続（H）
        return self.out(u)                                        # (B, n_class, H, W) 入力と同解像度（H,Wが4の倍数のとき）